<a href="https://colab.research.google.com/github/shatrill/data_analytics/blob/main/session_2_data_structures_governance_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2: Data Structures & Governance Lab

**Course:** Data Analytics Foundation (CompTIA Data+) | **CompTIA Domain:** 1.0 Data Concepts & Environments | 5.0 Data Governance, Quality & Controls

> Open this notebook in **Google Colab** (colab.research.google.com → File → Upload notebook). Tonight you will act as a **data steward**: load structured (CSV) and semi-structured (JSON) data, fix data types, and scrub personal information so the data is safe to share under GDPR/HIPAA-style rules.

## 🗺️ Your 45-Minute Lab Map
| Part | What you will do | Approx. time |
|---|---|---|
| Part 1 | Load a CSV file and a nested JSON array | 6 min |
| Part 2 | Convert data types (float, int, datetime, ordered category) | 9 min |
| Part 3 | Guided Example: build a data dictionary and avoid the ZIP-code trap | 8 min |
| Part 4 | Write functions to mask SSNs and hash emails | 9 min |
| Part 5 | Drop raw PII and export a compliant dataset | 3 min |
| Part 6 | 🧩 Guided Challenge: prepare support tickets for an outside vendor | 10 min |

**How to use this notebook:** read each text cell, then run the code cell beneath it with **Shift + Enter**.
Cells marked **✏️ Your Turn** have blanks (`___`) for you to fill in. Each one has a hidden solution
you can reveal *after* you have tried it. Errors are normal and part of learning. Read the last line
of the error message, fix it, and run the cell again.

In [ ]:
# Libraries for tonight
import pandas as pd      # tables (DataFrames)
import json              # reading JSON text
import hashlib           # one-way hashing for privacy

## Part 1: Loading Structured and Semi-Structured Data

| Structure | Looks like | Example formats |
|---|---|---|
| **Structured** | Strict rows and columns, like a filing cabinet with labeled folders | CSV, SQL tables, Excel |
| **Semi-structured** | Key-value pairs that can nest inside each other, like a stack of receipts with standard fields | JSON, XML |
| **Unstructured** | No fixed model at all | Images, audio, PDFs, free text |

### 1A. Load a CSV file
In real life you would download a CSV from a system. Here we first **create** a small CSV file in Colab's storage, then load it exactly as you would a real one.

In [ ]:
csv_text = '''customer_id,full_name,ssn,email,zip_code,signup_date,annual_spend,store_visits,satisfaction
1001,Maria Lopez,123-45-6789,maria.lopez@email.com,14623,2023-01-15,"$1,250.00",12,High
1002,James Chen,987-65-4321,JChen@Email.com,02134,2023-03-02,$980.50,8,Medium
1003,Aisha Patel,555-12-3456,aisha.p@email.com,07030,2022-11-20,"$2,410.75",20,High
1004,Robert King,222-33-4444,rking@email.com,14607,2024-02-10,$310.00,3,Low
1005,Emily Davis,777-88-9999,emily.davis@email.com,90210,2023-07-04,"$1,875.20",15,Medium
1006,Kwame Mensah,333-22-1111,kwame.m@email.com,10001,2024-05-18,$640.00,5,Low
'''
# Write the text to a real file named customers.csv
with open("customers.csv", "w") as f:
    f.write(csv_text)

customers = pd.read_csv("customers.csv")
customers

NameError: name 'pd' is not defined

### 1B. Load a nested JSON array
JSON (JavaScript Object Notation) is the language of web APIs. Each record is wrapped in curly braces `{ }` and holds **key: value** pairs. A value can itself be another `{ }` object. That is the "nesting."

`pd.json_normalize()` **flattens** nested JSON into a table. Nested keys become column names joined with a dot, such as `customer.email`.

In [ ]:
orders_json = '''
[
  {"order_id": "A-501", "order_date": "2024-03-05", "total": "89.99",
   "customer": {"id": 1001, "email": "maria.lopez@email.com"},
   "shipping": {"city": "Rochester", "state": "NY"}},
  {"order_id": "A-502", "order_date": "2024-03-06", "total": "240.00",
   "customer": {"id": 1003, "email": "aisha.p@email.com"},
   "shipping": {"city": "Hoboken", "state": "NJ"}},
  {"order_id": "A-503", "order_date": "2024-03-09", "total": "15.25",
   "customer": {"id": 1002, "email": "JChen@Email.com"},
   "shipping": {"city": "Boston", "state": "MA"}}
]
'''
records = json.loads(orders_json)       # text -> Python list of dictionaries
orders = pd.json_normalize(records)     # nested dictionaries -> flat table
orders

## Part 2: Converting Data Types

Computers treat `"1,250.00"` (text) and `1250.00` (a number) very differently, and you cannot average text. Let us see what pandas guessed for each column:

In [ ]:
customers.dtypes

`object` means text. `annual_spend` is text because of the `$` and `,` characters, `signup_date` is text, and `satisfaction` is plain text even though *Low < Medium < High* has a natural order.

### 2A. String → float (decimal number)
Remove the `$` and `,` characters, then convert with `.astype(float)`.

In [ ]:
customers["annual_spend"] = (customers["annual_spend"]
                             .str.replace("$", "", regex=False)
                             .str.replace(",", "", regex=False)
                             .astype(float))
customers["annual_spend"]

### 2B. Make sure counts are integers
`store_visits` is a **discrete** count (1, 2, 3...), so it should be a whole number.

In [ ]:
customers["store_visits"] = customers["store_visits"].astype(int)
print(customers["store_visits"].dtype, "| total visits:", customers["store_visits"].sum())

### 2C. String → datetime
Real dates let you sort chronologically, pull out the year or month, and do date math.

In [ ]:
customers["signup_date"] = pd.to_datetime(customers["signup_date"])

customers["signup_year"] = customers["signup_date"].dt.year       # pull out the year
customers["days_as_customer"] = (pd.Timestamp("2025-01-01") - customers["signup_date"]).dt.days
customers[["full_name", "signup_date", "signup_year", "days_as_customer"]]

### 2D. String → ordered category (ordinal data)
An **ordered categorical** tells pandas the ranking, so sorting and comparisons follow business logic instead of the alphabet. Alphabetically, "High" would come before "Low"!

In [ ]:
satisfaction_order = pd.CategoricalDtype(categories=["Low", "Medium", "High"], ordered=True)
customers["satisfaction"] = customers["satisfaction"].astype(satisfaction_order)

# Now comparisons respect the order
print(customers[customers["satisfaction"] >= "Medium"][["full_name", "satisfaction"]])
print()
print(customers.sort_values("satisfaction")[["full_name", "satisfaction"]])

In [ ]:
customers.dtypes   # check our work

## Part 3: Guided Example: Build a Data Dictionary (and the ZIP-Code Trap)

A **data dictionary** documents every column: its data type and its **measurement scale**. CompTIA loves testing this.

| Scale | Meaning | Example |
|---|---|---|
| **Nominal** | Labels, no order | name, email, ZIP code |
| **Ordinal** | Labels with a rank | satisfaction (Low/Medium/High) |
| **Discrete** | Countable whole numbers | store_visits |
| **Continuous** | Measured values, decimals allowed | annual_spend |

### Step 1: Spot the trap
Look at James Chen's ZIP code in the original file: `02134`. Now look at what pandas loaded:

In [ ]:
customers[["full_name", "zip_code"]]

Pandas saw digits and assumed a number, so **`02134` became `2134`** and `07030` became `7030`. Those are now invalid ZIP codes! This is a real-world data-quality bug.

**Why it happened:** a ZIP code *looks* numeric, but it is a **nominal label**. You would never add or average ZIP codes, so it should be stored as text.

### Step 2: Fix it at load time with `dtype=`

In [ ]:
fixed = pd.read_csv("customers.csv", dtype={"zip_code": str})
customers["zip_code"] = fixed["zip_code"]
customers[["full_name", "zip_code"]]

### Step 3: Generate the data dictionary
We build a small table automatically: one row per column, with its pandas type. Then we add the measurement scale using our analyst judgment. A computer cannot decide that part for us.

In [ ]:
scale = {
    "customer_id": "Nominal (identifier)", "full_name": "Nominal", "ssn": "Nominal (PII)",
    "email": "Nominal (PII)", "zip_code": "Nominal", "signup_date": "Interval (date)",
    "annual_spend": "Continuous", "store_visits": "Discrete", "satisfaction": "Ordinal",
    "signup_year": "Discrete", "days_as_customer": "Discrete",
}

data_dictionary = pd.DataFrame({
    "column": customers.columns,
    "pandas_dtype": customers.dtypes.astype(str).values,
    "measurement_scale": [scale.get(c, "?") for c in customers.columns],
    "example_value": customers.iloc[0].astype(str).values,
})
data_dictionary

**💬 Takeaway:** `customer_id` is stored as an integer, but it is a **nominal identifier**. Averaging customer IDs is meaningless. Data types describe *storage*; measurement scales describe *meaning*.

## Part 4: Protecting PII with Custom Functions

**PII** (Personally Identifiable Information) includes names, SSNs, emails and phone numbers. Two tools from tonight's lecture:

- **Masking** hides part of a value for *display*, for example `XXX-XX-6789`. A call-center agent can verify the last four digits without seeing the rest.
- **Hashing** turns a value into a scrambled fingerprint using a one-way math function. The same email always produces the same hash, so analysts can still count unique customers or join tables, but nobody can reverse it back to the email. This is a form of **pseudonymization**.

### 4A. A function to mask SSNs
A **function** is a reusable recipe. `def` starts it, the value goes in through the parentheses, and `return` sends the result back.

In [ ]:
def mask_ssn(ssn):
    """Return an SSN with only the last 4 digits visible, e.g. XXX-XX-6789."""
    return "XXX-XX-" + ssn[-4:]       # ssn[-4:] means "the last 4 characters"

print(mask_ssn("123-45-6789"))   # test on one value first

customers["ssn_masked"] = customers["ssn"].apply(mask_ssn)   # apply to every row
customers[["full_name", "ssn", "ssn_masked"]]

### 4B. A function to hash emails
Notice the emails `JChen@Email.com` and `jchen@email.com` refer to the same person. We **standardize** first with `.strip().lower()`, and then hash. Otherwise the same person would get two different fingerprints.

In [ ]:
def hash_email(email):
    """Standardize an email, then return its SHA-256 hash (first 16 characters for readability)."""
    clean = email.strip().lower()
    return hashlib.sha256(clean.encode("utf-8")).hexdigest()[:16]

print(hash_email("JChen@Email.com"))
print(hash_email("  jchen@email.com "))   # same person -> same hash

customers["email_hash"] = customers["email"].apply(hash_email)
customers[["email", "email_hash"]]

## Part 5: Create the Compliant Export

Now drop every **raw** PII column so only the protected versions remain. Then save the file.

In [ ]:
pii_columns = ["full_name", "ssn", "email"]

customers_safe = customers.drop(columns=pii_columns)
customers_safe.to_csv("customers_compliant.csv", index=False)

print("Columns in the export:", list(customers_safe.columns))
customers_safe.head()

📁 Click the **folder icon** on Colab's left sidebar to see `customers_compliant.csv`. Right-click it to download.

## Part 6: 🧩 Guided Challenge: Prepare Support Tickets for an Outside Vendor

Your company is sending customer-support tickets to an outside analytics vendor. Legal says the vendor may see the **issue details**, a **masked phone number**, a **salted email hash**, and an **age band**, but **no** names, raw emails, raw phone numbers, or exact birth dates.

Work through the five steps. Each one has a hint.

In [ ]:
tickets_json = '''
[
 {"ticket_id": "T-9001", "opened": "2025-02-01",
  "customer": {"name": "Maria Lopez", "email": "maria.lopez@email.com", "phone": "585-555-0142", "birth_date": "1988-04-12"},
  "issue": {"category": "Billing", "priority": "High"}},
 {"ticket_id": "T-9002", "opened": "2025-02-02",
  "customer": {"name": "James Chen", "email": "JChen@Email.com", "phone": "617-555-0199", "birth_date": "1975-09-30"},
  "issue": {"category": "Shipping", "priority": "Low"}},
 {"ticket_id": "T-9003", "opened": "2025-02-02",
  "customer": {"name": "Aisha Patel", "email": "aisha.p@email.com", "phone": "201-555-0107", "birth_date": "2001-01-22"},
  "issue": {"category": "Billing", "priority": "Medium"}},
 {"ticket_id": "T-9004", "opened": "2025-02-04",
  "customer": {"name": "Robert King", "email": "rking@email.com", "phone": "585-555-0175", "birth_date": "1962-06-03"},
  "issue": {"category": "Returns", "priority": "High"}}
]
'''
print("Ticket data ready.")

**Step 1: Flatten the JSON.** Load it with `json.loads()`, then flatten with `pd.json_normalize()`. Look at the new column names.

In [ ]:
tickets = pd.___(json.loads(tickets_json))
tickets.columns

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
tickets = pd.json_normalize(json.loads(tickets_json))
tickets.columns
```

</details>

**Step 2: Mask the phone number** so only the last 4 digits show, for example `XXX-XXX-0142`. Use your `mask_ssn` function as a model.

In [ ]:
def mask_phone(phone):
    return "XXX-XXX-" + phone[___]

tickets["phone_masked"] = tickets["customer.phone"].apply(___)
tickets[["customer.phone", "phone_masked"]]

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
def mask_phone(phone):
    return "XXX-XXX-" + phone[-4:]

tickets["phone_masked"] = tickets["customer.phone"].apply(mask_phone)
tickets[["customer.phone", "phone_masked"]]
```

</details>

**Step 3: Salted hash.** A plain hash of a common email can sometimes be guessed by hashing millions of known emails. A **salt** is a secret string added before hashing, so outsiders cannot recreate the hashes. Add the salt to the cleaned email, then hash.

In [ ]:
SECRET_SALT = "mcc-data-2025"     # in real life this lives in a secure vault, never in the notebook!

def salted_hash(email):
    clean = email.strip().lower()
    return hashlib.sha256((___ + clean).encode("utf-8")).hexdigest()[:16]

tickets["email_hash"] = tickets["customer.email"].apply(___)
tickets[["customer.email", "email_hash"]]

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
SECRET_SALT = "mcc-data-2025"

def salted_hash(email):
    clean = email.strip().lower()
    return hashlib.sha256((SECRET_SALT + clean).encode("utf-8")).hexdigest()[:16]

tickets["email_hash"] = tickets["customer.email"].apply(salted_hash)
tickets[["customer.email", "email_hash"]]
```

</details>

**Step 4: Generalize birth dates into age bands.** **Generalization** is an anonymization technique: an exact birth date is identifying, but "35–49" is not.
- Convert `customer.birth_date` to datetime.
- Compute age in years as of the ticket's `opened` date (days ÷ 365.25).
- Use `pd.cut()` to put ages into bands.

In [ ]:
tickets["customer.birth_date"] = pd.to_datetime(tickets["customer.birth_date"])
tickets["opened"] = pd.to_datetime(tickets["opened"])

tickets["age"] = ((tickets["opened"] - tickets["___"]).dt.days / 365.25).astype(int)

tickets["age_band"] = pd.cut(tickets["age"],
                             bins=[0, 24, 34, 49, 64, 120],
                             labels=["18-24", "25-34", "35-49", "50-64", "65+"])
tickets[["customer.birth_date", "age", "age_band"]]

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
tickets["customer.birth_date"] = pd.to_datetime(tickets["customer.birth_date"])
tickets["opened"] = pd.to_datetime(tickets["opened"])
tickets["age"] = ((tickets["opened"] - tickets["customer.birth_date"]).dt.days / 365.25).astype(int)
tickets["age_band"] = pd.cut(tickets["age"], bins=[0, 24, 34, 49, 64, 120],
                             labels=["18-24", "25-34", "35-49", "50-64", "65+"])
tickets[["customer.birth_date", "age", "age_band"]]
```

</details>

**Step 5: Build the vendor file and run an automated compliance check.** Drop every raw PII column, including the exact `age`, because it can help re-identify someone. Then let Python verify that no `@` signs or full phone numbers remain anywhere. `assert` stops with an error if a check fails.

In [ ]:
raw_pii = ["customer.name", "customer.email", "customer.phone", "customer.birth_date", "___"]
vendor = tickets.drop(columns=raw_pii)

# --- Automated compliance check ---
as_text = vendor.astype(str)
assert not as_text.apply(lambda col: col.str.contains("@")).any().any(), "Raw email found!"
assert not as_text.apply(lambda col: col.str.contains(r"\d{3}-\d{3}-\d{4}")).any().any(), "Raw phone found!"
print("✅ Compliance check passed. Columns shared with vendor:", list(vendor.columns))
vendor

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
raw_pii = ["customer.name", "customer.email", "customer.phone", "customer.birth_date", "age"]
vendor = tickets.drop(columns=raw_pii)

as_text = vendor.astype(str)
assert not as_text.apply(lambda col: col.str.contains("@")).any().any(), "Raw email found!"
assert not as_text.apply(lambda col: col.str.contains(r"\d{3}-\d{3}-\d{4}")).any().any(), "Raw phone found!"
print("✅ Compliance check passed. Columns shared with vendor:", list(vendor.columns))
vendor
```

</details>

**✍️ Reflect (double-click to answer):** Which steps were *pseudonymization* (reversible if you hold the key, or still linkable), and which were *anonymization* (not traceable)? Why might the vendor still be able to link two tickets from the same customer?

<details>
<summary><b>💡 Reveal a sample answer</b></summary>

- **Pseudonymization:** the salted email hash. Anyone holding the secret salt can re-create the hash and link it back, and the same customer always gets the same hash.
- **Masking:** the phone number. It is partially hidden for display.
- **Anonymization (generalization):** age bands replace exact birth dates.
- The vendor can link two tickets from the same customer because the email hash is **consistent**. That is intentional: it lets them count repeat customers without knowing who they are.

</details>

---
## ✅ Lab Checklist
- [ ] Loaded a CSV and flattened nested JSON with `json_normalize`
- [ ] Converted strings to float, int, datetime and an ordered category
- [ ] Built a data dictionary and fixed the ZIP-code trap
- [ ] Masked SSNs and hashed emails with my own functions
- [ ] Produced a compliant export and passed an automated compliance check

## 💾 Save Your Work to GitHub
1. In Colab, click **File → Save a copy in GitHub**.
2. Choose your course/Capstone repository.
3. Use this commit message: `Session 2: Completed Data Structures & Governance Lab`

**Coming up next:** Session 3: relational databases and your first SQL queries.